# Lesson 4 — PCW Code: Decision Trees

Organized from the PCW code and starter-code screenshot discussed in “决策树知识讲解”.

This notebook calculates weighted Gini impurity, searches both features for the best first split, and repeats the search on each child. Candidate thresholds are observed unique values, as required by the exercise; left means `<=`, right means `>`.

**Data requirement:** the original `x_data` and `y_data` arrays and their generation code were not available in the conversation. Supply them in Section 2. No substitute dataset or fabricated execution results are included.

Requires Python 3 and NumPy (`pip install numpy`).

## 1. Gini impurity and split search

The starter functions are preserved in purpose, with input validation and handling for empty/unsplittable subsets. Empty-child candidate splits are excluded. Ties retain the first candidate (axis order, then ascending threshold).

In [ ]:
import numpy as np


def validate_data(x_data, y_data):
    """Validate the two-feature, binary-label data used by this PCW."""
    x_data = np.asarray(x_data, dtype=float)
    y_data = np.asarray(y_data)
    if x_data.ndim != 2 or x_data.shape[1] != 2:
        raise ValueError("x_data must have shape (n_samples, 2).")
    if y_data.ndim != 1 or len(x_data) != len(y_data):
        raise ValueError("y_data must be one-dimensional and match x_data.")
    if not np.isfinite(x_data).all():
        raise ValueError("Feature values must be finite.")
    if not np.isin(y_data, [0, 1]).all():
        raise ValueError("This PCW expects binary labels 0 and 1.")
    return x_data, y_data


def leaf_gini_impurity(leaf_ys):
    """Return binary Gini impurity; an empty leaf contributes zero."""
    leaf_ys = np.asarray(leaf_ys)
    if len(leaf_ys) == 0:
        return 0.0
    p0 = np.mean(leaf_ys == 0)
    p1 = np.mean(leaf_ys == 1)
    return float(1 - p0**2 - p1**2)


def weighted_gini_impurity(left_ys, right_ys):
    """Weight each child's impurity by its sample count."""
    total = len(left_ys) + len(right_ys)
    if total == 0:
        return 0.0
    return (
        len(left_ys) * leaf_gini_impurity(left_ys)
        + len(right_ys) * leaf_gini_impurity(right_ys)
    ) / total


def gini_impurity(x_data, y_data, axis, threshold):
    """Return weighted impurity for a proposed feature/threshold split."""
    left_mask = x_data[:, axis] <= threshold
    return weighted_gini_impurity(y_data[left_mask], y_data[~left_mask])


def find_best_split(x_subset, y_subset):
    """Return (axis, threshold, weighted_gini), or None if unsplittable."""
    x_subset, y_subset = validate_data(x_subset, y_subset)
    if len(y_subset) < 2 or leaf_gini_impurity(y_subset) == 0:
        return None

    best_gini = float("inf")
    best_split = None
    for axis in [0, 1]:
        for threshold in np.unique(x_subset[:, axis]):
            left_mask = x_subset[:, axis] <= threshold
            if not left_mask.any() or left_mask.all():
                continue
            gini = gini_impurity(x_subset, y_subset, axis, threshold)
            if gini < best_gini:
                best_gini = gini
                best_split = (axis, float(threshold), float(gini))
    return best_split


def print_split(name, split):
    print(name)
    if split is None:
        print("No split: this subset is pure or has no valid threshold.")
    else:
        axis, threshold, gini = split
        print(f"Best axis: {axis} ({'X' if axis == 0 else 'Y'}-axis)")
        print(f"Best threshold: {threshold:.16g}")
        print(f"Weighted Gini impurity: {gini:.16g}")


## 2. Supply the original PCW data

Run the original course data-generation cell here to define `x_data` (two columns) and `y_data` (labels 0/1). Alternatively, load a local NumPy archive containing those two arrays. Uncomment and adjust the example below. The following sections are skipped until both arrays exist.

In [ ]:
# Paste the original PCW data-generation code here.

# Alternatively:
# with np.load("lesson4_data.npz", allow_pickle=False) as data:
#     x_data = data["x_data"]
#     y_data = data["y_data"]

data_ready = "x_data" in globals() and "y_data" in globals()
if data_ready:
    x_data, y_data = validate_data(x_data, y_data)
    if len(y_data) == 0:
        raise ValueError("Supply a nonempty PCW dataset.")
    print(f"Loaded {len(y_data)} observations.")
else:
    print("Original PCW data missing: define x_data and y_data, then rerun from here.")


## 3. Find the first split

Search all unique values on both the X-axis (0) and Y-axis (1).

In [ ]:
first_split = None
if data_ready:
    first_split = find_best_split(x_data, y_data)
    print_split("First split:", first_split)


## 4. Find the second split in each child

Use the computed first split rather than hard-coding its threshold. For an axis-1 first split, 'left' is the lower-Y subset and 'right' is the upper-Y subset.

In [ ]:
if data_ready and first_split is not None:
    first_axis, first_threshold, _ = first_split
    left_mask = x_data[:, first_axis] <= first_threshold
    right_mask = ~left_mask

    x_left, y_left = x_data[left_mask], y_data[left_mask]
    x_right, y_right = x_data[right_mask], y_data[right_mask]

    left_split = find_best_split(x_left, y_left)
    right_split = find_best_split(x_right, y_right)

    print_split("Left child:", left_split)
    print()
    print_split("Right child:", right_split)


## 5. Results recorded in the original discussion

These are historical results from the conversation, **not outputs recomputed in this notebook**. Reproducing them requires the same original dataset.

| Node | Axis | Threshold | Weighted Gini |
| --- | --- | --- | --- |
| First split | 1 (Y) | 0.06998012608535217 | 0.27036666666666664 |
| Left child | 0 (X) | -0.7612104943756963 | 0.09481865284974082 |
| Right child | 0 (X) | 1.2443389030267236 | 0.20788104089219325 |

Thresholds are feature values and can be negative or greater than 1. Binary Gini impurity ranges from 0 to 0.5.